In [ ]:
!pip install -q gradio scikit-learn pandas numpy matplotlib

: 

In [ ]:
import re
import random
import datetime
import html

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import gradio as gr

random.seed(42)
np.random.seed(42)

In [ ]:
products = [
    "Paracetamol 500mg Tablet",
    "Amoxicillin 250mg Capsule",
    "Metformin 500mg Tablet",
    "Insulin Glargine Injection",
    "Omeprazole 20mg Capsule",
    "Salbutamol Inhaler",
    "Cetirizine Syrup",
    "Amlodipine 5mg Tablet"
]

customer_types = [
    "Patient",
    "Pharmacist",
    "Hospital",
    "Physician",
    "Distributor"
]

categories = [
    "Product Quality",
    "Packaging Defect",
    "Labeling Issue",
    "Contamination",
    "Stability/Potency",
    "Adverse Event",
    "Supply/Logistics",
    "Efficacy Concern"
]

severity_levels = ["Critical", "Major", "Minor"]

severity_weights_by_category = {
    "Product Quality": [0.10, 0.60, 0.30],
    "Packaging Defect": [0.15, 0.55, 0.30],
    "Labeling Issue": [0.25, 0.55, 0.20],
    "Contamination": [0.75, 0.20, 0.05],
    "Stability/Potency": [0.45, 0.45, 0.10],
    "Adverse Event": [0.75, 0.20, 0.05],
    "Supply/Logistics": [0.15, 0.55, 0.30],
    "Efficacy Concern": [0.25, 0.60, 0.15],
}

category_templates = {
    "Product Quality": [
        "Customer received {product} from batch {batch} with chipped, cracked, or broken tablets.",
        "Tablets in {product} batch {batch} appear discolored and have unusual spots.",
        "Capsules in {product} batch {batch} are sticking together and some are empty.",
        "Patient found damaged dosage units in {product} batch {batch}.",
        "Pharmacist reported crumbling tablets while dispensing {product} batch {batch}."
    ],
    "Packaging Defect": [
        "Blister pack of {product} batch {batch} has broken seal and exposed tablets.",
        "Carton of {product} batch {batch} was crushed and primary pack is damaged.",
        "Missing patient information leaflet inside pack of {product} batch {batch}.",
        "Child-resistant cap on {product} batch {batch} is not closing properly.",
        "Leaking bottle observed for {product} batch {batch} during pharmacy receipt."
    ],
    "Labeling Issue": [
        "Expiry date printed on {product} batch {batch} label is smudged and unreadable.",
        "Label claim on {product} batch {batch} does not match the product inside.",
        "Wrong strength mentioned on carton of {product} batch {batch}.",
        "Batch number and expiry mismatch on label of {product} batch {batch}.",
        "Missing warning sticker on {product} batch {batch} pack."
    ],
    "Contamination": [
        "Foreign black particles found in {product} batch {batch}.",
        "Visible mold or microbial growth suspected in {product} batch {batch}.",
        "Unusual chemical odor noticed when opening {product} batch {batch}.",
        "Glass fragments reported near container of {product} batch {batch}.",
        "Hair or fiber contamination observed in {product} batch {batch} pack."
    ],
    "Stability/Potency": [
        "{product} batch {batch} was exposed to high temperature during storage.",
        "Pharmacist suspects potency loss because patients report poor response to {product} batch {batch}.",
        "Cold chain temperature excursion occurred for {product} batch {batch}.",
        "Product {product} batch {batch} changed color after storage.",
        "Dissolution concern raised for {product} batch {batch} after routine checks."
    ],
    "Adverse Event": [
        "Patient experienced severe rash and breathing difficulty after taking {product} batch {batch}.",
        "Hospitalization reported due to suspected reaction to {product} batch {batch}.",
        "Patient reports unexpected bleeding after using {product} batch {batch}.",
        "Overdose risk because patient misunderstood dosing of {product} batch {batch}.",
        "Serious allergic reaction suspected with {product} batch {batch}."
    ],
    "Supply/Logistics": [
        "Shipment of {product} batch {batch} arrived later than required and temperature log missing.",
        "Wrong quantity delivered for {product} batch {batch}.",
        "Pallet of {product} batch {batch} was dropped during warehouse handling.",
        "Distributor received damaged outer cases of {product} batch {batch}.",
        "Cold chain shipment for {product} batch {batch} showed temperature excursion."
    ],
    "Efficacy Concern": [
        "Patient states {product} batch {batch} is no longer controlling blood glucose.",
        "Physician reports lack of therapeutic response with {product} batch {batch}.",
        "Patient says blood pressure remains high despite taking {product} batch {batch}.",
        "Pharmacist received multiple complaints that {product} batch {batch} seems ineffective.",
        "Patient switched from {product} batch {batch} and symptoms improved, suggesting product issue."
    ]
}

severity_phrases = {
    "Critical": [
        "This is a serious patient safety issue requiring immediate escalation.",
        "Potential recall and regulatory notification may be needed.",
        "Risk of severe harm, hospitalization, or death.",
        "Urgent quality event with high regulatory impact."
    ],
    "Major": [
        "This is a significant GMP deviation requiring formal investigation.",
        "Potential nonconformance and CAPA required.",
        "Multiple units or patients may be affected.",
        "Requires quality review before further distribution."
    ],
    "Minor": [
        "This appears to be a minor cosmetic issue with low patient risk.",
        "Single isolated occurrence with limited impact.",
        "No immediate patient safety concern identified.",
        "Can be handled through routine quality review."
    ]
}


def generate_dataset(n=3000):
    rows = []

    for _ in range(n):
        category = random.choice(categories)
        product = random.choice(products)
        batch = f"LOT{random.randint(10000, 99999)}"

        template = random.choice(category_templates[category])
        text = template.format(product=product, batch=batch)

        weights = severity_weights_by_category[category]
        severity = random.choices(severity_levels, weights=weights, k=1)[0]

        text += " " + random.choice(severity_phrases[severity])

        rows.append({
            "text": text,
            "category": category,
            "severity": severity
        })

    return pd.DataFrame(rows)


df = generate_dataset(3000)
df.head()

In [ ]:
X = df["text"]
y_cat = df["category"]
y_sev = df["severity"]

X_train, X_test, y_cat_train, y_cat_test, y_sev_train, y_sev_test = train_test_split(
    X,
    y_cat,
    y_sev,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])

severity_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])

category_model.fit(X_train, y_cat_train)
severity_model.fit(X_train, y_sev_train)

cat_pred = category_model.predict(X_test)
sev_pred = severity_model.predict(X_test)

cat_acc = accuracy_score(y_cat_test, cat_pred)
sev_acc = accuracy_score(y_sev_test, sev_pred)

print(f"Category accuracy: {cat_acc:.4f}")
print(f"Severity accuracy: {sev_acc:.4f}")

print("\nCategory classification report:")
print(classification_report(y_cat_test, cat_pred))

print("\nSeverity classification report:")
print(classification_report(y_sev_test, sev_pred))

metrics_df = pd.DataFrame({
    "Model": ["Category Classifier", "Severity Classifier"],
    "Test Accuracy": [round(cat_acc, 4), round(sev_acc, 4)],
    "Purpose": [
        "Predict complaint type from text",
        "Predict complaint severity from text"
    ]
})

metrics_df

In [ ]:
queue_columns = [
    "ID",
    "Date",
    "Product",
    "Batch",
    "Customer Type",
    "Category",
    "Severity",
    "Priority Level",
    "Priority Score",
    "Status",
    "Owner"
]

all_columns = [
    "ID",
    "Date",
    "Description",
    "Product",
    "Batch",
    "Customer Type",
    "Reporter",
    "Category",
    "Severity",
    "Sentiment",
    "Priority Score",
    "Priority Level",
    "Status",
    "Owner"
]

complaints_df = pd.DataFrame(columns=all_columns)

CAPA_ACTIONS = {
    "Adverse Event": [
        "Notify Pharmacovigilance and Safety team within 24 hours.",
        "Collect patient outcome, concomitant medications, and medical history.",
        "Assess need for expedited regulatory reporting.",
        "Quarantine retained samples and review batch records."
    ],
    "Contamination": [
        "Immediately quarantine affected batch and distribution stock.",
        "Initiate microbiological or foreign matter testing.",
        "Review environmental monitoring, line clearance, and filtration records.",
        "Assess batch recall risk and notify Quality Assurance."
    ],
    "Product Quality": [
        "Retain samples and perform visual/physical testing.",
        "Review manufacturing, compression/coating, and in-process controls.",
        "Inspect transport and storage conditions.",
        "Determine if OOS investigation or CAPA is required."
    ],
    "Packaging Defect": [
        "Check packaging line records, seal integrity, and cartoning operations.",
        "Perform AQL inspection of retained samples.",
        "Review transport damage and secondary packaging.",
        "Correct labeling/artwork if involved."
    ],
    "Labeling Issue": [
        "Verify label version, expiry, batch overprinting, and artwork approval.",
        "Check labeling line reconciliation and line clearance.",
        "Assess risk of medication error and notify Regulatory Affairs.",
        "Issue market notification if labeling is noncompliant."
    ],
    "Stability/Potency": [
        "Initiate OOS/stability investigation.",
        "Test retained samples for assay, dissolution, and degradation products.",
        "Review storage conditions and cold chain data.",
        "Assess shelf-life and possible recall."
    ],
    "Supply/Logistics": [
        "Review distribution records, temperature logs, and chain of custody.",
        "Investigate carrier handling and cold chain excursion.",
        "Assess product impact and replace/credit if needed.",
        "Implement preventive controls with logistics partner."
    ],
    "Efficacy Concern": [
        "Verify authenticity, storage, and expiry.",
        "Review batch release and stability data.",
        "Collect clinical details and concomitant therapy.",
        "Evaluate possible subpotency or drug interaction."
    ]
}

owner_map = {
    "Adverse Event": "Pharmacovigilance",
    "Contamination": "Quality Assurance / Microbiology",
    "Product Quality": "Quality Assurance / Production",
    "Packaging Defect": "Packaging QA",
    "Labeling Issue": "Regulatory Affairs / QA",
    "Stability/Potency": "Quality Control / Stability",
    "Supply/Logistics": "Supply Chain QA",
    "Efficacy Concern": "Medical Affairs / QA"
}

product_keywords = {
    "paracetamol": "Paracetamol 500mg Tablet",
    "acetaminophen": "Paracetamol 500mg Tablet",
    "amoxicillin": "Amoxicillin 250mg Capsule",
    "metformin": "Metformin 500mg Tablet",
    "insulin": "Insulin Glargine Injection",
    "glargine": "Insulin Glargine Injection",
    "omeprazole": "Omeprazole 20mg Capsule",
    "salbutamol": "Salbutamol Inhaler",
    "albuterol": "Salbutamol Inhaler",
    "cetirizine": "Cetirizine Syrup",
    "amlodipine": "Amlodipine 5mg Tablet"
}

negative_words = [
    "angry", "upset", "disappointed", "danger", "unsafe", "severe",
    "serious", "urgent", "hospital", "injury", "death", "allergic",
    "contamination", "wrong", "damaged", "broken", "defective",
    "expired", "leak", "ineffective", "failure", "concern", "risk", "adverse"
]

positive_words = [
    "good", "satisfied", "excellent", "thank", "no issue", "resolved"
]

flag_terms = {
    "Adverse event": ["adverse", "allergic", "reaction", "hospital", "injury", "death", "overdose"],
    "Contamination": ["contamination", "foreign", "particle", "mold", "microbial", "glass", "hair"],
    "Labeling risk": ["wrong label", "wrong strength", "wrong product", "expiry", "unreadable", "missing leaflet"],
    "Recall potential": ["recall", "market action", "serious patient safety"],
    "Cold chain": ["cold chain", "temperature excursion", "temperature log"]
}


def detect_product(text):
    if not text:
        return None

    t = text.lower()

    for key, value in product_keywords.items():
        if key in t:
            return value

    return None


def extract_batch(text):
    if not text:
        return None

    patterns = [
        r"batch\s*[:#-]?\s*([A-Za-z0-9]{4,20})",
        r"lot\s*[:#-]?\s*([A-Za-z0-9]{4,20})",
        r"\b(LOT\d{4,10})\b",
        r"\b(B\d{4,10})\b"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).upper()

    return None


def sentiment_label(text):
    t = text.lower()

    neg_count = sum(1 for word in negative_words if word in t)
    pos_count = sum(1 for word in positive_words if word in t)

    score = neg_count - pos_count

    if score >= 2:
        return "Negative"
    elif score <= -1:
        return "Positive"
    else:
        return "Neutral"


def regulatory_flags(text):
    t = text.lower()
    flags = []

    for label, terms in flag_terms.items():
        if any(term in t for term in terms):
            flags.append(label)

    return flags


def calculate_priority(severity, sentiment, customer_type, flags):
    base_score = {
        "Critical": 80,
        "Major": 55,
        "Minor": 25
    }.get(severity, 35)

    sentiment_score = {
        "Negative": 10,
        "Neutral": 5,
        "Positive": 0
    }.get(sentiment, 0)

    customer_score = {
        "Hospital": 12,
        "Physician": 12,
        "Patient": 10,
        "Pharmacist": 8,
        "Distributor": 6
    }.get(customer_type, 4)

    flag_score = min(len(flags) * 5, 15)

    total_score = base_score + sentiment_score + customer_score + flag_score
    total_score = min(total_score, 100)

    if total_score >= 85:
        level = "Immediate"
    elif total_score >= 70:
        level = "High"
    elif total_score >= 45:
        level = "Medium"
    else:
        level = "Low"

    return {
        "score": int(total_score),
        "level": level
    }


def top_predictions(model, text, n=3):
    probabilities = model.predict_proba([text])[0]
    classes = model.classes_

    top_items = sorted(
        zip(classes, probabilities),
        key=lambda x: x[1],
        reverse=True
    )[:n]

    return [(str(cls), float(prob)) for cls, prob in top_items]


def get_queue_df(severity_filter="All", category_filter="All"):
    if complaints_df.empty:
        return pd.DataFrame(columns=queue_columns)

    filtered_df = complaints_df.copy()

    if severity_filter != "All":
        filtered_df = filtered_df[filtered_df["Severity"] == severity_filter]

    if category_filter != "All":
        filtered_df = filtered_df[filtered_df["Category"] == category_filter]

    if filtered_df.empty:
        return pd.DataFrame(columns=queue_columns)

    filtered_df = filtered_df.sort_values("Date", ascending=False)

    return filtered_df[queue_columns].reset_index(drop=True)


def render_analysis_html(analysis):
    e = html.escape

    top_categories = "".join([
        f"<li>{e(name)}: {prob:.1%}</li>"
        for name, prob in analysis["top_categories"]
    ])

    top_severities = "".join([
        f"<li>{e(name)}: {prob:.1%}</li>"
        for name, prob in analysis["top_severities"]
    ])

    flags_html = "".join([
        f"<li>{e(flag)}</li>"
        for flag in analysis["flags"]
    ]) or "<li>No major regulatory flags detected.</li>"

    capa_html = "".join([
        f"<li>{e(action)}</li>"
        for action in analysis["capa"]
    ])

    severity_color = {
        "Critical": "#ffcdd2",
        "Major": "#ffe0b2",
        "Minor": "#c8e6c9"
    }.get(analysis["severity"], "#e0e0e0")

    return f"""
    <div style="border:1px solid #d0d7de; border-radius:12px; padding:18px; background:#ffffff; font-family: Arial, sans-serif;">
        <h2 style="margin-top:0;">AI Complaint Assessment</h2>

        <p>
            <strong>Complaint ID:</strong> {e(analysis['id'])}<br>
            <strong>Date:</strong> {e(analysis['date'])}<br>
            <strong>Product:</strong> {e(analysis['product'])}<br>
            <strong>Batch/Lot:</strong> {e(analysis['batch'])}<br>
            <strong>Customer Type:</strong> {e(analysis['customer_type'])}<br>
            <strong>Reporter:</strong> {e(analysis['reporter'])}
        </p>

        <p>
            <strong>AI Category:</strong>
            <span style="background:#e3f2fd; padding:4px 10px; border-radius:8px; font-weight:bold;">
                {e(analysis['category'])}
            </span>
        </p>

        <p>
            <strong>AI Severity:</strong>
            <span style="background:{severity_color}; padding:4px 10px; border-radius:8px; font-weight:bold;">
                {e(analysis['severity'])}
            </span>
        </p>

        <p>
            <strong>Sentiment:</strong> {e(analysis['sentiment'])}<br>
            <strong>Priority:</strong> {e(analysis['priority_level'])} ({analysis['priority_score']}/100)<br>
            <strong>Suggested Owner:</strong> {e(analysis['owner'])}
        </p>

        <h3>Top Category Predictions</h3>
        <ul>{top_categories}</ul>

        <h3>Top Severity Predictions</h3>
        <ul>{top_severities}</ul>

        <h3>Regulatory / Quality Flags</h3>
        <ul>{flags_html}</ul>

        <h3>Suggested CAPA Actions</h3>
        <ul>{capa_html}</ul>
    </div>
    """


def analyze_and_submit(text, product, batch, customer_type, reporter, severity_filter, category_filter):
    global complaints_df

    if not text or not text.strip():
        fig1, fig2 = make_charts()
        return (
            "⚠️ Please enter a complaint description.",
            "<p style='color:red;'>No complaint text provided.</p>",
            get_queue_df(severity_filter, category_filter),
            fig1,
            fig2
        )

    top_categories = top_predictions(category_model, text, n=3)
    top_severities = top_predictions(severity_model, text, n=3)

    predicted_category = top_categories[0][0]
    predicted_severity = top_severities[0][0]

    final_product = product if product != "Auto-detect" else (detect_product(text) or "Unspecified")

    final_batch = batch.strip() if batch and batch.strip() else (extract_batch(text) or "Not provided")

    sentiment = sentiment_label(text)
    flags = regulatory_flags(text)
    priority = calculate_priority(predicted_severity, sentiment, customer_type, flags)

    owner = owner_map.get(predicted_category, "Quality Assurance")
    capa = CAPA_ACTIONS.get(predicted_category, ["Initiate quality investigation."])

    complaint_id = f"CMP-{datetime.datetime.now().year}-{len(complaints_df)+1:04d}"
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    row = {
        "ID": complaint_id,
        "Date": now,
        "Description": text.strip(),
        "Product": final_product,
        "Batch": final_batch,
        "Customer Type": customer_type,
        "Reporter": reporter or "Not provided",
        "Category": predicted_category,
        "Severity": predicted_severity,
        "Sentiment": sentiment,
        "Priority Score": priority["score"],
        "Priority Level": priority["level"],
        "Status": "Open",
        "Owner": owner
    }

    complaints_df = pd.concat(
        [complaints_df, pd.DataFrame([row])],
        ignore_index=True
    )

    analysis = {
        "id": complaint_id,
        "date": now,
        "product": final_product,
        "batch": final_batch,
        "customer_type": customer_type,
        "reporter": reporter or "Not provided",
        "category": predicted_category,
        "severity": predicted_severity,
        "sentiment": sentiment,
        "priority_score": priority["score"],
        "priority_level": priority["level"],
        "owner": owner,
        "flags": flags,
        "capa": capa,
        "top_categories": top_categories,
        "top_severities": top_severities
    }

    status_message = (
        f"✅ **{complaint_id}** saved.  \n"
        f"AI Triage: **{predicted_category}** | Severity: **{predicted_severity}** | "
        f"Priority: **{priority['level']}** ({priority['score']}/100)"
    )

    fig1, fig2 = make_charts()

    return (
        status_message,
        render_analysis_html(analysis),
        get_queue_df(severity_filter, category_filter),
        fig1,
        fig2
    )


def create_empty_figure(message):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=12)
    ax.axis("off")
    return fig


def make_charts():
    if complaints_df.empty:
        return create_empty_figure("No complaints submitted yet"), create_empty_figure("No complaints submitted yet")

    # Category chart
    fig1, ax1 = plt.subplots(figsize=(7, 4))
    category_counts = complaints_df["Category"].value_counts()
    category_counts.plot(kind="bar", ax=ax1, color="steelblue")
    ax1.set_title("Complaints by Category")
    ax1.set_xlabel("Category")
    ax1.set_ylabel("Count")
    plt.setp(ax1.get_xticklabels(), rotation=25, ha="right")
    fig1.tight_layout()

    # Severity chart
    fig2, ax2 = plt.subplots(figsize=(5, 4))

    severity_order = [
        severity for severity in ["Critical", "Major", "Minor"]
        if severity in complaints_df["Severity"].values
    ]

    severity_counts = complaints_df["Severity"].value_counts().reindex(severity_order)

    severity_counts.plot(
        kind="bar",
        ax=ax2,
        color=["#d9534f", "#f0ad4e", "#5cb85c"][:len(severity_order)]
    )

    ax2.set_title("Complaints by Severity")
    ax2.set_xlabel("Severity")
    ax2.set_ylabel("Count")
    fig2.tight_layout()

    return fig1, fig2


initial_cat_plot, initial_sev_plot = make_charts()

In [ ]:
explanation_md = """
## How this AI system works

### 1. Text input
The user enters a customer complaint in natural language.

### 2. AI classification
Two machine learning models predict:
- Complaint category
- Complaint severity

### 3. Rule-based enhancement
The system also applies business rules to:
- detect product name
- extract batch/lot number
- detect sentiment
- generate regulatory flags
- calculate priority score
- suggest responsible department
- suggest CAPA actions

### 4. Dashboard
The complaint is stored in an in-memory complaint queue and shown in the dashboard.

### Note
This demo uses synthetic training data and in-memory storage.
For production, use a secure database, authentication, audit logs, validated models, and pharma compliance controls.
"""

with gr.Blocks(title="AI Pharma Complaint Management System") as demo:
    gr.Markdown("""
    # AI-Powered Customer Complaint Management System
    ### Pharmaceutical Manufacturing Intern Project Demo
    """)

    with gr.Tabs():

        with gr.Tab("📝 New Complaint"):
            with gr.Row():
                with gr.Column(scale=2):
                    complaint_text = gr.Textbox(
                        label="Complaint Description",
                        lines=6,
                        placeholder="Example: Hospital received multiple blister packs of Amoxicillin 250mg Capsule LOT56789 with broken seals and missing leaflets. This is a serious packaging issue and may affect patient safety."
                    )

                    product_input = gr.Dropdown(
                        ["Auto-detect"] + products,
                        label="Product",
                        value="Auto-detect"
                    )

                    batch_input = gr.Textbox(
                        label="Batch/Lot Number",
                        placeholder="LOT12345"
                    )

                    customer_type_input = gr.Dropdown(
                        customer_types,
                        label="Customer/Reporter Type",
                        value="Patient"
                    )

                    reporter_input = gr.Textbox(
                        label="Reporter Name / ID (optional)",
                        placeholder="Enter reporter name or ticket ID"
                    )

                    submit_btn = gr.Button("AI Analyze & Submit", variant="primary")

                    gr.Markdown("""
                    **Try this example:**

                    Hospital received multiple blister packs of Amoxicillin 250mg Capsule LOT56789 with broken seals and missing leaflets. This is a serious packaging issue and may affect patient safety.
                    """)

                with gr.Column(scale=3):
                    status_msg = gr.Markdown("")
                    analysis_html = gr.HTML("")

        with gr.Tab("📋 Complaint Queue"):
            with gr.Row():
                filter_severity = gr.Dropdown(
                    ["All"] + severity_levels,
                    label="Filter by Severity",
                    value="All"
                )

                filter_category = gr.Dropdown(
                    ["All"] + categories,
                    label="Filter by Category",
                    value="All"
                )

                refresh_btn = gr.Button("Refresh Queue")

            queue_table = gr.Dataframe(
                value=get_queue_df("All", "All"),
                interactive=False,
                label="Complaint Queue"
            )

        with gr.Tab("📊 Analytics"):
            analytics_refresh = gr.Button("Refresh Analytics")

            with gr.Row():
                cat_plot = gr.Plot(label="Complaints by Category", value=initial_cat_plot)
                sev_plot = gr.Plot(label="Complaints by Severity", value=initial_sev_plot)

            metrics_table = gr.Dataframe(
                value=metrics_df,
                label="Model Performance",
                interactive=False
            )

            gr.Markdown(f"**Category accuracy:** {cat_acc:.2%} | **Severity accuracy:** {sev_acc:.2%}")

        with gr.Tab("ℹ️ How It Works"):
            gr.Markdown(explanation_md)

    submit_btn.click(
        fn=analyze_and_submit,
        inputs=[
            complaint_text,
            product_input,
            batch_input,
            customer_type_input,
            reporter_input,
            filter_severity,
            filter_category
        ],
        outputs=[
            status_msg,
            analysis_html,
            queue_table,
            cat_plot,
            sev_plot
        ]
    )

    refresh_btn.click(
        fn=get_queue_df,
        inputs=[filter_severity, filter_category],
        outputs=queue_table
    )

    analytics_refresh.click(
        fn=make_charts,
        inputs=None,
        outputs=[cat_plot, sev_plot]
    )

demo.launch(share=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')